# Kabyle Dataset Exploration

Explore the Mozilla Common Voice Kabyle dataset: statistics, audio samples, and text distribution.

In [ ]:
import os

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from dotenv import load_dotenv

from src.data.dataset import KabyleDataset

load_dotenv()

## 1. Load the Dataset

In [ ]:
dataset = KabyleDataset(hf_token=os.getenv("HF_TOKEN"))
data = dataset.load()

print("Splits and sizes:")
for split, count in dataset.get_stats().items():
    print(f"  {split}: {count:,} examples")

## 2. Sample Examples

In [ ]:
train = data["train"]
print("Column names:", train.column_names)
print("\nFirst 5 sentences:")
for i in range(5):
    print(f"  [{i}] {train[i]['sentence']}")

## 3. Audio Duration Distribution

In [ ]:
# Compute durations for a subset
num_samples = min(1000, len(train))
durations = []
for i in range(num_samples):
    audio = train[i]["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    durations.append(duration)

durations = np.array(durations)
print(f"Duration stats (first {num_samples} samples):")
print(f"  Mean: {durations.mean():.2f}s")
print(f"  Std:  {durations.std():.2f}s")
print(f"  Min:  {durations.min():.2f}s")
print(f"  Max:  {durations.max():.2f}s")

plt.figure(figsize=(10, 4))
plt.hist(durations, bins=50, edgecolor="black")
plt.xlabel("Duration (seconds)")
plt.ylabel("Count")
plt.title("Audio Duration Distribution")
plt.tight_layout()
plt.show()

## 4. Visualize Audio Waveform and Spectrogram

In [ ]:
sample = train[0]
audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Waveform
axes[0].set_title(f"Waveform: \"{sample['sentence']}\"")
librosa.display.waveshow(audio_array, sr=sr, ax=axes[0])
axes[0].set_xlabel("Time (s)")

# Mel spectrogram
S = librosa.feature.melspectrogram(y=audio_array, sr=sr, n_mels=80)
S_dB = librosa.power_to_db(S, ref=np.max)
librosa.display.specshow(S_dB, sr=sr, x_axis="time", y_axis="mel", ax=axes[1])
axes[1].set_title("Mel Spectrogram")

plt.tight_layout()
plt.show()

## 5. Text Statistics

In [ ]:
sentences = [train[i]["sentence"] for i in range(num_samples)]
word_counts = [len(s.split()) for s in sentences]
char_counts = [len(s) for s in sentences]

print(f"Text stats (first {num_samples} samples):")
print(f"  Avg words per sentence: {np.mean(word_counts):.1f}")
print(f"  Avg chars per sentence: {np.mean(char_counts):.1f}")

# Character frequency
all_chars = Counter("".join(sentences).lower())
print(f"  Unique characters: {len(all_chars)}")
print(f"  Most common: {all_chars.most_common(15)}")